# I. Data Curation

### Goal:
Read in Amazon product reviews metadata from multiple categories, clean and standardize descriptions, scrub identifiers, handle product weights, balance the categories to fix class imbalance, and push the final curated datasets back to the Hugging Face Hub.

In [ ]:
# Install required libraries on Google Colab
!pip install huggingface_hub datasets==3.6.0 pydantic tqdm matplotlib numpy python-dotenv

In [ ]:
# Imports
import os
import re
import json
import random
from datetime import datetime
from typing import Optional
from concurrent.futures import ProcessPoolExecutor

from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np

In [ ]:
# HuggingFace Hub Authentication
# Set your HF_TOKEN in your Google Colab Secrets (the key icon in the left menu)
# or paste it directly below.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', 'YOUR_HF_TOKEN_HERE')

if hf_token and hf_token != 'YOUR_HF_TOKEN_HERE':
    login(hf_token, add_to_git_credential=True)
else:
    print("WARNING: HF_TOKEN is not set. You will need to log in to push datasets to the Hub.")

## 1. Utilities (Items, Parser, Loaders)

Here we define all the utility classes and functions inline, matching what was in the `pricer` package.

In [ ]:
# ========================================== #
# 1A. Item Schema Definition
# ========================================== #
from pydantic import BaseModel

PREFIX = "Price is $"
QUESTION = "What does this cost to the nearest dollar?"

class Item(BaseModel):
    """
    An Item is a data-point of a Product with a Price
    """
    title: str
    category: str
    price: float
    full: Optional[str] = None
    weight: Optional[float] = None
    summary: Optional[str] = None
    prompt: Optional[str] = None
    id: Optional[int] = None

    def make_prompt(self, text: str):
        self.prompt = f"{QUESTION}\n\n{text}\n\n{PREFIX}{round(self.price)}.00"

    def test_prompt(self) -> str:
        return self.prompt.split(PREFIX)[0] + PREFIX

    def __repr__(self) -> str:
        return f"<{self.title} = ${self.price}>"

    @staticmethod
    def push_to_hub(dataset_name: str, train: list["Item"], val: list["Item"], test: list["Item"]):
        """Push Item lists to HuggingFace Hub"""
        DatasetDict(
            {
                "train": Dataset.from_list([item.model_dump() for item in train]),
                "validation": Dataset.from_list([item.model_dump() for item in val]),
                "test": Dataset.from_list([item.model_dump() for item in test]),
            }
        ).push_to_hub(dataset_name)

    @classmethod
    def from_hub(cls, dataset_name: str) -> tuple[list["Item"], list["Item"], list["Item"]]:
        """Load from HuggingFace Hub and reconstruct Items"""
        ds = load_dataset(dataset_name)
        return (
            [cls.model_validate(row) for row in ds["train"]],
            [cls.model_validate(row) for row in ds["validation"]],
            [cls.model_validate(row) for row in ds["test"]],
        )

# ========================================== #
# 1B. Scrubbing & Parsing Functions
# ========================================== #
MIN_CHARS = 600
MIN_PRICE = 0.5
MAX_PRICE = 999.49
MAX_TEXT_EACH = 3000
MAX_TEXT_TOTAL = 4000

REMOVALS = [
    "Part Number",
    "Best Sellers Rank",
    "Batteries Included?",
    "Batteries Required?",
    "Item model number",
]

def simplify(text_list) -> str:
    """
    Return a simplified string without too much whitespace and limited characters
    """
    return (
        str(text_list)
        .replace("\n", " ")
        .replace("\r", "")
        .replace("\t", "")
        .replace("  ", " ")
        .strip()[:MAX_TEXT_EACH]
    )

def scrub(title, description, features, details) -> str:
    """
    Return a cleansed full string with product numbers and unimportant details removed
    """
    for remove in REMOVALS:
        details.pop(remove, None)
    result = title + "\n"
    if description:
        result += simplify(description) + "\n"
    if features:
        result += simplify(features) + "\n"
    if details:
        result += json.dumps(details) + "\n"
    pattern = r"\b(?=[A-Z0-9]{7,}\b)(?=.*[A-Z])(?=.*\d)[A-Z0-9]+\b"
    return re.sub(pattern, "", result).strip()[:MAX_TEXT_TOTAL]

def get_weight(details):
    """
    Normalize the product weight into pounds
    """
    weight_str = details.get("Item Weight")
    if weight_str:
        parts = weight_str.split(" ")
        amount = float(parts[0])
        unit = parts[1].lower()
        if unit == "pounds":
            return amount
        elif unit == "ounces":
            return amount / 16
        elif unit == "grams":
            return amount / 453.592
        elif unit == "milligrams":
            return amount / 453592
        elif unit == "kilograms":
            return amount / 0.453592
        elif unit == "hundredths" and parts[2].lower() == "pounds":
            return amount / 100
    return 0

def parse(datapoint, category):
    """
    Parse a raw Amazon metadata datapoint dictionary into an Item
    """
    try:
        price = float(datapoint["price"])
    except ValueError:
        return None
    if MIN_PRICE <= price <= MAX_PRICE:
        title = datapoint["title"]
        description = datapoint["description"]
        features = datapoint["features"]
        try:
            details = json.loads(datapoint["details"])
        except Exception:
            details = {}
        weight = get_weight(details)
        full = scrub(title, description, features, details)
        if len(full) >= MIN_CHARS:
            return Item(
                title=title,
                category=category,
                price=price,
                full=full,
                weight=weight,
            )

# ========================================== #
# 1C. Parallel Processing Loader
# ========================================== #
CHUNK_SIZE = 1000
cpu_count = os.cpu_count() or 2
WORKERS = max(cpu_count - 1, 1)

class ItemLoader:
    def __init__(self, category):
        self.category = category
        self.dataset = None

    def from_datapoint(self, datapoint):
        return parse(datapoint, self.category)

    def from_chunk(self, chunk):
        batch = [self.from_datapoint(datapoint) for datapoint in chunk]
        return [item for item in batch if item is not None]

    def chunk_generator(self):
        size = len(self.dataset)
        for i in range(0, size, CHUNK_SIZE):
            yield self.dataset.select(range(i, min(i + CHUNK_SIZE, size)))

    def load_in_parallel(self, workers):
        results = []
        chunk_count = (len(self.dataset) // CHUNK_SIZE) + 1
        with ProcessPoolExecutor(max_workers=workers) as pool:
            for batch in tqdm(pool.map(self.from_chunk, self.chunk_generator()), total=chunk_count):
                results.extend(batch)
        return results

    def load(self, workers=WORKERS):
        start = datetime.now()
        print(f"Loading dataset {self.category}", flush=True)
        self.dataset = load_dataset(
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_meta_{self.category}",
            split="full",
            trust_remote_code=True,
        )
        results = self.load_in_parallel(workers)
        finish = datetime.now()
        print(
            f"Completed {self.category} with {len(results):,} datapoints in {(finish - start).total_seconds() / 60:.1f} mins",
            flush=True,
        )
        return results

## 2. Load and Explore a Single Dataset (Appliances)

In [ ]:
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)
print(f"Number of Appliances: {len(dataset):,}")

In [ ]:
# Investigate a particular datapoint
dataset[6]

In [ ]:
# Find the most expensive item
max_price = 0
max_item = None

for datapoint in tqdm(dataset):
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except ValueError:
        pass

print(f"The most expensive item is {max_item['title']} and it costs {max_price:,.2f}")

Most expensive item looks like a commercial oven. Let's parse all items using single-threaded parse first.

In [ ]:
# Load into Item objects if they have a price range $1-$1000 and enough details
items = [parse(datapoint, "Appliances") for datapoint in tqdm(dataset)]
items = [item for item in items if item is not None]
print(f"There are {len(items):,} items from {len(dataset):,} datapoints")

In [ ]:
items[0]

In [ ]:
print(items[0].full)

In [ ]:
prices = [item.price for item in items]
lengths = [len(item.full) for item in items]

In [ ]:
# Plot the distribution of lengths
plt.figure(figsize=(15, 6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 6000, 100))
plt.show()

In [ ]:
max_length = max(lengths)
max_length_item = items[lengths.index(max_length)]
print(max_length_item.full)

In [ ]:
# Plot the distribution of prices
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()

In [ ]:
print(items[3].full)

## 3. Load Multiple Categories in Parallel

We now use the custom `ItemLoader` to pull and clean 8 categories.

In [ ]:
# Load single category to test loader
loader = ItemLoader("Appliances")
items = loader.load()

In [ ]:
dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]

In [ ]:
items = []
for dataset_name in dataset_names:
    loader = ItemLoader(dataset_name)
    items.extend(loader.load())

In [ ]:
print(f"A grand total of {len(items):,} items")

In [ ]:
items[1000]

In [ ]:
random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication, we have {len(items):,} items")

In [ ]:
lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.1f} and highest {max(lengths):,}\n")
plt.xlabel('Length (characters)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()

In [ ]:
# Plot the distribution of prices
prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
from collections import Counter
category_counts = Counter([item.category for item in items])

categories = list(category_counts.keys())
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()

## 4. Class Balancing & Weighted Downsampling

Let's balance the representation of different categories so that Tools and Automotive don't overly dominate.

In [ ]:
np.random.seed(42)

SIZE = 820_000

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p**2
w[categories == "Tools_and_Home_Improvement"] *= 0.5
w[categories == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
# Just for good measure, let's shuffle the sample again for the final dataset
random.seed(42)
random.shuffle(sample)

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
category_counts = Counter([item.category for item in sample])

categories = list(category_counts.keys())
counts = [category_counts[category] for category in categories]

# Bar chart by category
plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct='%1.0f%%', startangle=90)

centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Categories')
plt.axis('equal')  
plt.show()

In [ ]:
# How does the price vary with the character count?
sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")
plt.xlabel('Size')
plt.ylabel('Price')
plt.title('Is there a simple correlation with text length?')
plt.show()

In [ ]:
# How does the price vary with the weight?
ounces = [item.weight for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(ounces, prices, s=0.2, color="darkorange")
plt.xlabel('Weight (ounces)')
plt.ylabel('Price')
plt.xlim(0, 400)
plt.title('Is there a simple correlation with weight?')
plt.show()

## 5. Push to HuggingFace Hub

Remember to change the `username` below to your own HuggingFace username before running.

In [ ]:
username = "your-hf-username"
lite = f"{username}/items_raw_lite"

train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

print(f"Pushing lite dataset splits (train={len(train_lite):,}, val={len(val_lite):,}, test={len(test_lite):,}) to {lite}...")
try:
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)
except Exception as e:
    print(f"Error pushing lite dataset: {e}.")

# II. Data Pre-processing (LLM-based Standardization)

Instead of feeding raw, verbose, and noisy Amazon product descriptions directly into downstream models,standardize the data by using an LLM to rewrite each product description into a structured, token-efficient format.

### Key Operations:
1. **Load Raw Dataset**: Loads the raw curated dataset (`items_raw_lite`) from the Hugging Face Hub.
2. **LLM Standardization Schema**: Runs product text through an LLM (`openai/gpt-oss-20b` or local `ollama/llama3.2`) with a system prompt to format each product description into a clean, 5-line summary structure:
   - **Title**: A short, precise title (removing part numbers).
   - **Category**: Standardized category name (e.g., Electronics).
   - **Brand**: Cleansed brand name.
   - **Description**: A single-sentence product description.
   - **Details**: A single-sentence summary of key features.
3. **Groq Batch API Execution**:
   - To make processing fast and cost-effective, products are split into batches of 1,000 and formatted into JSONL request files.
   - Batches are submitted asynchronously using the **Groq Batch API** and then retrieved once completed.
4. **Cleanup & Hub Push**:
   - The retrieved LLM summaries are mapped back to each item's `summary` field.
   - Raw verbose text fields (`full`) and temporary item IDs (`id`) are removed to save hub space.
   - The final standardized datasets are split and pushed to the Hugging Face Hub as `items_lite` (20,000 train items).
